In [3]:
import pandas as pd

sales = pd.read_csv("train.csv", parse_dates=["date"])

print("Dimensions :", sales.shape)
print("Date minimale :", sales["date"].min())
print("Date maximale :", sales["date"].max())
print("Nombre de magasins :", sales["store"].nunique())
print("Nombre d’articles :", sales["item"].nunique())

sales.head()

Dimensions : (913000, 4)
Date minimale : 2013-01-01 00:00:00
Date maximale : 2017-12-31 00:00:00
Nombre de magasins : 10
Nombre d’articles : 50


,date,store,item,sales
0,2013-01-01,1,1,13
1,2013-01-02,1,1,11
2,2013-01-03,1,1,14
3,2013-01-04,1,1,13
4,2013-01-05,1,1,10


In [2]:
from google.colab import files

uploaded = files.upload()

Saving train.csv to train.csv


In [4]:
# Historical data available before the forecast period
sales_train = sales.loc[sales["date"] < "2017-10-01"].copy()

# Actual sales used to evaluate the forecasts
sales_test = sales.loc[
    sales["date"].between("2017-10-01", "2017-12-31")
].copy()

for name, data in [("Training", sales_train), ("Evaluation", sales_test)]:
    print(
        name,
        "| Rows:", len(data),
        "| From:", data["date"].min().date(),
        "| To:", data["date"].max().date()
    )

Training | Rows: 867000 | From: 2013-01-01 | To: 2017-09-30
Evaluation | Rows: 46000 | From: 2017-10-01 | To: 2017-12-31


In [5]:
# Check missing values in each column
print("Missing values:")
print(sales.isna().sum())

# Each date–store–item combination should appear once
duplicate_count = sales.duplicated(
    subset=["date", "store", "item"]
).sum()

print("\nDuplicate date–store–item combinations:", duplicate_count)
print("Minimum sales:", sales["sales"].min())
print("Maximum sales:", sales["sales"].max())

Missing values:
date     0
store    0
item     0
sales    0
dtype: int64

Duplicate date–store–item combinations: 0
Minimum sales: 0
Maximum sales: 231


In [6]:
def add_calendar_features(data):
    result = data.copy()

    result["year"] = result["date"].dt.year
    result["month"] = result["date"].dt.month
    result["day_of_week"] = result["date"].dt.dayofweek
    result["day_of_month"] = result["date"].dt.day
    result["time_index"] = (
        result["date"] - pd.Timestamp("2013-01-01")
    ).dt.days

    return result


train_features = add_calendar_features(sales_train)
test_features = add_calendar_features(sales_test)

train_features.head()

,date,store,item,sales,year,month,day_of_week,day_of_month,time_index
0,2013-01-01,1,1,13,2013,1,1,1,0
1,2013-01-02,1,1,11,2013,1,2,2,1
2,2013-01-03,1,1,14,2013,1,3,3,2
3,2013-01-04,1,1,13,2013,1,4,4,3
4,2013-01-05,1,1,10,2013,1,5,5,4


In [7]:
feature_columns = [
    "store",
    "item",
    "year",
    "month",
    "day_of_week",
    "day_of_month",
    "time_index"
]

X_train = train_features[feature_columns]
y_train = train_features["sales"]

X_test = test_features[feature_columns]
y_test = test_features["sales"]

print("Training inputs:", X_train.shape)
print("Evaluation inputs:", X_test.shape)

Training inputs: (867000, 7)
Evaluation inputs: (46000, 7)


In [9]:
from sklearn.ensemble import HistGradientBoostingRegressor

model = HistGradientBoostingRegressor(
    max_iter=200,
    learning_rate=0.1,
    max_leaf_nodes=31,
    early_stopping=False,
    random_state=42
)

model.fit(X_train, y_train)

print("Model training completed.")

Model training completed.


In [10]:
from sklearn.metrics import mean_absolute_error

# Forecast the evaluation period
predictions = model.predict(X_test).clip(min=0)

# Calculate errors using forecast minus actual sales
mae_python = mean_absolute_error(y_test, predictions)
mean_error_python = (predictions - y_test.to_numpy()).mean()

print(f"Python model MAE: {mae_python:.2f}")
print(f"Python model Mean Error: {mean_error_python:.2f}")

Python model MAE: 6.33
Python model Mean Error: 0.66


In [11]:
forecast_results = sales_test[
    ["date", "store", "item", "sales"]
].copy()

forecast_results = forecast_results.rename(
    columns={
        "date": "ForecastDate",
        "sales": "ActualSales"
    }
)

forecast_results["ForecastSales_Python"] = predictions

forecast_results["Error"] = (
    forecast_results["ForecastSales_Python"]
    - forecast_results["ActualSales"]
)

forecast_results["AbsoluteError"] = forecast_results["Error"].abs()

forecast_results.head()

,ForecastDate,store,item,ActualSales,ForecastSales_Python,Error,AbsoluteError
1734,2017-10-01,1,1,21,26.574948,5.574948,5.574948
1735,2017-10-02,1,1,12,17.118233,5.118233,5.118233
1736,2017-10-03,1,1,18,19.612242,1.612242,1.612242
1737,2017-10-04,1,1,15,19.612242,4.612242,4.612242
1738,2017-10-05,1,1,20,20.321100,0.321100,0.321100


In [12]:
from google.colab import files

output_file = "Forecast_Evaluation_Python.csv"

forecast_results.to_csv(
    output_file,
    index=False,
    date_format="%Y-%m-%d"
)

files.download(output_file)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>